# 05. Evaluate and calibrate

A decision model returns probabilities, so accuracy alone is not enough. We also ask whether the probabilities are calibrated: when the model says 0.8, should it be right about 80 percent of the time?

We compute four things: accuracy, Brier score, expected calibration error, and a confidence threshold that escalates uncertain decisions. All code cells are small and end with a validation.


## Step 1. Load the dataset and a client

We use the mock so the notebook is reproducible without a server. The mock makes a wrong choice 25 percent of the time, which lets us measure accuracy and calibration honestly. Swap in a real client later; the evaluation code does not change.


In [ ]:
from decision_qa import MockClient, read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
client = MockClient(seed=42, quality=0.75, error_rate=0.25)
print(type(client).__name__)


## Step 2. One result, one probability

`gold_probability` extracts the probability the model assigned to the correct answer. We demonstrate it on a single question before looping.


In [ ]:
from decision_qa import gold_label, gold_probability, predicted_label
record = records[0]
question = record.questions[0]
result = client.decision(record.state, [question])[0]
print(result.probabilities)


In [ ]:
p = gold_probability(result, question)
print(round(p, 3))
assert 0.0 <= p <= 1.0


In [ ]:
print(gold_label(question))
print(predicted_label(result))


## Step 3. Collect all predictions

We walk through every record and every question. For each answer we store three things: the probability of the gold answer, the model's top label, and the confidence of that top label. Calibration is always computed on the predicted confidence, not on the gold one.


In [ ]:
gold_probs = []
confidences = []
correct = []
all_results = []
all_questions = []
for rec in records:
    results = client.decision(rec.state, rec.questions)
    all_results.extend(results)
    all_questions.extend(rec.questions)
    for q, res in zip(rec.questions, results):
        gold_probs.append(gold_probability(res, q))
        confidences.append(max(res.probabilities.values()))
        correct.append(int(predicted_label(res) == gold_label(q)))


In [ ]:
print(len(gold_probs), len(confidences), len(correct))
assert len(gold_probs) == len(confidences) == len(correct)


## Step 4. Accuracy

Accuracy is the share of questions where the gold answer was the top answer. It says nothing about the quality of the probability values.


In [ ]:
acc = sum(correct) / len(correct)
print(round(acc, 3))
assert 0.0 <= acc <= 1.0


In [ ]:
from decision_qa import accuracy
library_acc = accuracy(all_results, all_questions)
print(round(library_acc, 3))
assert abs(library_acc - acc) < 1e-9


## Step 5. Brier score

Brier score is the mean squared error of the gold probabilities. Lower is better. A model that is both accurate and confident scores close to zero.


In [ ]:
from decision_qa import brier
score = brier(gold_probs)
print(round(score, 3))
assert 0.0 <= score <= 1.0


In [ ]:
one = brier([0.8])
print(round(one, 3))
assert abs(one - 0.04) < 1e-9


In [ ]:
from decision_qa import brier_multiclass
multiclass = brier_multiclass(all_results, all_questions)
print(round(multiclass, 3))
assert 0.0 <= multiclass <= 2.0


## Step 6. Reliability and ECE

The reliability table groups predictions by confidence and compares average confidence with observed accuracy. ECE is the weighted gap between the two.


In [ ]:
from decision_qa import reliability
rows = reliability(confidences, correct)
print(len(rows))


In [ ]:
for row in rows:
    print(row['bin'], row['count'], round(row['confidence'], 2), round(row['accuracy'], 2))


The reliability diagram plots observed accuracy against predicted confidence. Points on the dashed diagonal are perfectly calibrated; points below it mean the model is overconfident. The gap is what ECE summarizes as a single number.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration')
centers = [row['confidence'] for row in rows]
accuracies = [row['accuracy'] for row in rows]
ax.plot(centers, accuracies, 'o-', color='#E45756', label='model')
ax.set_xlabel('Predicted confidence')
ax.set_ylabel('Observed accuracy')
ax.set_title('Reliability diagram')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
plt.show()


In [ ]:
from decision_qa import ece
error = ece(confidences, correct)
print(round(error, 3))
assert 0.0 <= error <= 1.0


## Step 7. A confidence threshold

`fit_threshold` finds the lowest cutoff that reaches a target precision on this data. Decisions below the cutoff should be escalated. In a real project the threshold must be fit on a validation split, never on the test split.


In [ ]:
from decision_qa import fit_threshold
threshold = fit_threshold(confidences, correct, target_precision=0.9)
print(round(threshold, 2))


In [ ]:
coverage = sum(p >= threshold for p in confidences) / len(confidences)
print(round(coverage, 3))
assert 0.0 <= coverage <= 1.0


## Step 8. Option order invariance

Finally we check that a labelled choice keeps the same top answer when options are reordered. We compare a reliable model with the noisy one: the reliable model should be stable, while the noisy model can fail the check. This guards against positional bias.


In [ ]:
from decision_qa import ChoiceQuestion, MockClient
from decision_qa.evaluate import option_order_invariance
q = ChoiceQuestion(label='check', options=['a', 'b', 'c'], gold_index=1)
stable = MockClient(seed=5)


In [ ]:
print('stable model:', option_order_invariance(stable, {}, q))


In [ ]:
print('noisy model :', option_order_invariance(client, {}, q))


In [ ]:
assert option_order_invariance(stable, {}, q) is True
print('the reliable model is stable under reordering')


## Reading the results critically

High accuracy with poor calibration means the rankings are useful but the probabilities should not be trusted as numbers. Free references worth reading: JevBench under MIT, ProbabilityBench, the paper Evaluating and Benchmarking the System One model, the Early Evidence Audit with its 14-item checklist, and Beyond Calibration. JevAdvBench reminds us that the state is untrusted input, and Jev Does Not Play Dice shows how badly probabilities can drift from reality.
